# PCA on returns — short window (last month)

A **short window** (last month) with **many stocks** ($\sim$500). This lands deep in the $N \gg T$ regime: hundreds of stocks, only ~20 days. The sample covariance is rank-deficient and $\Sigma^{-1}$ doesn't exist — the regime where you *must* use a factor model or shrinkage, and where random-matrix theory (Marchenko-Pastur) separates signal eigenvalues from noise.

We slice the last month from the cached 2-year prices (instant, no re-download) and use `top_n=500` (also cached).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stonks import get_prices, to_returns

%matplotlib inline


## Parameters

`MONTHS` is the window length, sliced from the cached 2-year prices.


In [ ]:
TOP_N = 500
MONTHS = 1
PERIOD = "2y"     # cached; we slice the last MONTHS from it
INTERVAL = "1d"


## Fetch and slice the last month


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field="close")
last = pd.Timestamp(prices.columns[-1])
cut = last - pd.DateOffset(months=MONTHS)
prices_win = prices.loc[:, prices.columns >= cut]

returns = to_returns(prices_win).dropna()
N, T = returns.shape
print(f"window: {prices_win.shape[1]} price days -> {T} daily returns")
print(f"N={N} stocks, T={T} days   (N > T: {'YES' if N > T else 'no'})")
returns


## The $N > T$ regime

With $N$ stocks and $T < N$ days, the $N\times N$ covariance has rank $\le T-1 \ll N$. It's **singular**, so $\Sigma^{-1}$ (and the closed-form Markowitz weights) don't exist. This is exactly why one needs a factor model ($\Sigma = BB^\top + \Psi$, low-rank + idiosyncratic) or shrinkage here.


In [ ]:
X = returns.to_numpy(dtype=float)
mu = X.mean(axis=1)
Xc = X - mu[:, None]
Sigma = (Xc @ Xc.T) / (T - 1)
print(f"cov shape {Sigma.shape}, rank {np.linalg.matrix_rank(Sigma)} (= T-1={T-1})")
print("cond(Sigma): %.1e (effectively singular)" % np.linalg.cond(Sigma))


## Signal vs noise: Marchenko-Pastur

For a random (noise) correlation matrix of $N$ variables over $T$ samples, the Marchenko-Pastur law puts eigenvalues in $[\lambda_-, \lambda_+]$ with $\lambda_\pm = (1 \pm \sqrt{N/T})^2$. Eigenvalues **above** $\lambda_+$ are signal (real factors); inside the bulk are noise; and $N-T$ of them are ~0 (the null space).


In [ ]:
# correlation matrix: standardize each stock's returns to unit variance
std = X.std(axis=1, keepdims=True)
Z = (X - mu[:, None]) / std
R = (Z @ Z.T) / T
ev = np.sort(np.linalg.eigvalsh(R))[::-1]

lam_minus = (1 - np.sqrt(N / T)) ** 2
lam_plus  = (1 + np.sqrt(N / T)) ** 2
n_signal = int((ev > lam_plus).sum())
n_noise  = int(((ev > 1e-9) & (ev <= lam_plus)).sum())
n_zero   = int((ev <= 1e-9).sum())
print(f"MP bulk: [{lam_minus:.2f}, {lam_plus:.2f}]")
print(f"signal(>MP+): {n_signal} | noise(in bulk): {n_noise} | ~zero(null space): {n_zero}")
print(f"signal eigenvalues: {np.round(ev[:n_signal], 2)}")


In [ ]:
m = n_signal + n_noise   # count of nonzero eigenvalues
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(np.arange(1, m + 1), ev[:m], "o-", ms=4)
ax.axhline(lam_plus, color="r", ls="--", label=rf"MP upper $\lambda_+$={lam_plus:.1f}")
ax.axhline(lam_minus, color="r", ls=":", label=rf"MP lower $\lambda_-$={lam_minus:.1f}")
ax.set_xlabel("eigenvalue index"); ax.set_ylabel("eigenvalue (correlation matrix)")
ax.set_title(f"{n_signal} signal eigenvalues above the MP noise bulk ({N} stocks, {T} days)")
ax.legend()


### Takeaway

In the $N \gg T$ regime almost everything is noise: of a $458\times458$ correlation matrix, only ~3 eigenvalues clear the Marchenko-Pastur cutoff (the market factor + a couple of sector factors); the rest sit in the noise bulk or the null space. So a sensible model of these returns is **low-rank** — a handful of factors plus idiosyncratic noise ($r = Bf + \epsilon$), the direct analog of PNMF. The sample covariance is unusable on its own (singular), but a factor-model $\Sigma = BB^\top + \Psi$ is well-conditioned and invertible even here.
